# Curso: Bioestatística — Distribuição Normal
### Autores: Sandro da Silva Camargo e Fernando Cardoso

Considerando que a distribuição de altura do brasileiro segue uma distribuição normal, a altura média é 174cm, com desvio padrão de 7cm, responda
*  Qual a probabilidade de alguém ter mais de 190cm?
*  Qual a probabilidade de alguém ter menos de 150cm?
*  Qual a probabilidade de alguém ter entre 190 e 200cm?


Abra este código no seu google colab clicando [aqui](http://colab.research.google.com/github/Sandrocamargo/biostatistics/blob/master/python/bioe_02_normaldist.ipynb).

In [ ]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

# ==========================================
# 1. FUNÇÕES DE CÁLCULO (LÓGICA PURA)
# ==========================================


def obter_probabilidade_maior(media, desvio, altura_alvo):
    """Calcula P(X > altura_alvo)."""
    prob = stats.norm.sf(altura_alvo, loc=media, scale=desvio)
    z_score = (altura_alvo - media) / desvio
    return prob, z_score


def obter_probabilidade_intervalo(media, desvio, altura_min, altura_max):
    """Calcula P(altura_min < X < altura_max)."""
    cdf_max = stats.norm.cdf(altura_max, loc=media, scale=desvio)
    cdf_min = stats.norm.cdf(altura_min, loc=media, scale=desvio)
    return cdf_max - cdf_min


# ==========================================
# 2. FUNÇÕES VISUAIS (GRÁFICOS)
# ==========================================


def plotar_distribuicao(
    media, desvio, limites, titulo="Distribuição Normal", rotulo_area=""
):
    """Gera o gráfico da curva normal com a área destacada."""
    x = np.linspace(media - 4 * desvio, media + 4 * desvio, 1000)
    y = stats.norm.pdf(x, loc=media, scale=desvio)

    plt.figure(figsize=(9, 5))
    plt.plot(x, y, color="navy", linewidth=2, label="Distribuição Normal")

    # Tratamento para cauda superior vs intervalo
    if len(limites) == 1:
        x_sombra = np.linspace(limites[0], media + 4 * desvio, 500)
        plt.axvline(
            x=limites[0],
            color="red",
            linestyle="--",
            label=f"Corte: {limites[0]} cm",
        )
    else:
        x_sombra = np.linspace(limites[0], limites[1], 500)
        plt.axvline(
            x=limites[0],
            color="green",
            linestyle="--",
            label=f"Mínimo: {limites[0]} cm",
        )
        plt.axvline(
            x=limites[1],
            color="green",
            linestyle="--",
            label=f"Máximo: {limites[1]} cm",
        )

    y_sombra = stats.norm.pdf(x_sombra, loc=media, scale=desvio)
    plt.fill_between(x_sombra, y_sombra, color="crimson", alpha=0.5, label=rotulo_area)

    plt.title(f"{titulo}\n(µ = {media} cm, σ = {desvio} cm)", fontsize=12)
    plt.xlabel("Altura (cm)", fontsize=10)
    plt.ylabel("Densidade de Probabilidade", fontsize=10)
    plt.legend(loc="upper left")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.show()


# ==========================================
# 3. TRATAMENTO DE ENTRADAS
# ==========================================


def ler_float(mensagem):
    """Garante a leitura correta de números com vírgula ou ponto."""
    while True:
        try:
            entrada = input(mensagem).replace(",", ".").strip()
            valor = float(entrada)
            return valor
        except ValueError:
            print("Entrada inválida. Por favor, digite um número válido.")


def obter_parametros_populacao():
    """Lê a média e o desvio padrão validando valores positivos."""
    print("\n--- Parâmetros da População ---")
    media = ler_float("Digite a média da população (cm): ")
    while True:
        desvio = ler_float("Digite o desvio padrão (cm): ")
        if desvio > 0:
            break
        print("O desvio padrão deve ser maior que zero.")
    return media, desvio


# ==========================================
# 4. MENU E FLUXO PRINCIPAL
# ==========================================


def menu():
    print("\n" + "=" * 45)
    print("      CALCULADORA DE BIOESTATÍSTICA")
    print("=" * 45)

    media, desvio = obter_parametros_populacao()

    while True:
        print("\nEscolha uma opção:")
        print("1. Probabilidade de altura MAIOR que um valor P(X > a)")
        print("2. Probabilidade de altura ENTRE dois valores P(a < X < b)")
        print("3. Alterar média e desvio padrão")
        print("4. Sair")

        opcao = input("\nOpção desejada (1-4): ").strip()

        if opcao == "1":
            alvo = ler_float("\nDigite a altura limite (cm): ")
            prob, z = obter_probabilidade_maior(media, desvio, alvo)
            pct = prob * 100

            print(f"\n-> Z-score: {z:.4f}")
            print(f"-> P(X > {alvo} cm) = {prob:.4f} ({pct:.2f}%)")

            plotar_distribuicao(
                media,
                desvio,
                limites=[alvo],
                titulo=f"Probabilidade de Altura > {alvo} cm",
                rotulo_area=f"P(X > {alvo}) = {pct:.2f}%",
            )

        elif opcao == "2":
            h_min = ler_float("\nDigite a altura MÍNIMA (cm): ")
            h_max = ler_float("Digite a altura MÁXIMA (cm): ")

            if h_min > h_max:
                h_min, h_max = h_max, h_min

            prob = obter_probabilidade_intervalo(media, desvio, h_min, h_max)
            pct = prob * 100

            print(f"\n-> P({h_min} < X < {h_max} cm) = {prob:.4f} ({pct:.2f}%)")

            plotar_distribuicao(
                media,
                desvio,
                limites=[h_min, h_max],
                titulo=f"Probabilidade entre {h_min} cm e {h_max} cm",
                rotulo_area=f"P({h_min} < X < {h_max}) = {pct:.2f}%",
            )

        elif opcao == "3":
            media, desvio = obter_parametros_populacao()

        elif opcao == "4":
            print("\nEncerrando o programa.")
            sys.exit(0)

        else:
            print("Opção inválida. Tente novamente.")

In [ ]:
if __name__ == "__main__":
    menu()